In [1]:
import shutil
import time

import win32com.client  # Windows COM 인터페이스 사용
import xml.etree.ElementTree as ET
import pandas as pd

import os.path

# 입력 파라미터
###################################################################################################################
network_path = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\시나리오 분석\시나리오(70-유고 100 간격)_260410\유고O\31~60"
network_file_name = r"화성~서울(70-랜덤 유고, 유고지점 추가)_260408.inpx"
network_full_path = os.path.join(network_path, network_file_name)

copy_network_path = os.path.join(network_path, f"화성~서울(70-랜덤 유고)_260414.inpx")

csv_path = r"C:\Users\(주)내일이엔시 도로교통안전연구소\Desktop\프로젝트\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\1. 회의자료\25.12.23"
csv_file_name = r"시나리오 테이블(영향권X).csv"

# 유고 발생 시각
acc_start_time = 3600
###################################################################################################################

csv_full_path  = os.path.join(csv_path, csv_file_name)

# 시나리오 파일 읽기
senario_df = pd.read_csv(csv_full_path, encoding="cp949")

Vissim = win32com.client.Dispatch("Vissim.Vissim")  # Vissim 인스턴스 생성

for index, row in senario_df.iloc[30:60].iterrows():

    # 교통량
    Q_main_in_vehph = row["Q_main_in_vehph"]
    Q_ramp_in_per_ramp = row["Q_ramp_in_per_ramp"]

    # 램프 유출 비율
    Q1_ramp_out_vehph = row["Q1_ramp_out_vehph"]
    Q2_ramp_out_vehph = row["Q2_ramp_out_vehph"]
    Q3_ramp_out_vehph = row["Q3_ramp_out_vehph"]

    # 본선 유입 중차량 비율
    V1_main_in_ratio = row["V1_main_in_ratio"]
    V2_main_in_ratio = row["V2_main_in_ratio"]
    V3_main_in_ratio = row["V3_main_in_ratio"]
    V4_main_in_ratio = row["V4_main_in_ratio"]
    V5_main_in_ratio = max(row["V5_main_in_ratio"], 0.001)

    # 램프 유입 중차량 비율
    V1_ramp_in_ratio = row["V1_ramp_in_ratio"]
    V2_ramp_in_ratio = row["V2_ramp_in_ratio"]
    V3_ramp_in_ratio = row["V3_ramp_in_ratio"]
    V4_ramp_in_ratio = row["V4_ramp_in_ratio"]
    V5_ramp_in_ratio = max(row["V5_ramp_in_ratio"], 0.001)

    # 종단경사
    #grade_percent = row["grade_percent"] * 0.01

    # 유고 지속시간
    incident_duration_min = row["incident_duration_min"]

    # 유고 차로수
    lane_closure_count = int(row["lane_closure_count"])

    # 랜덤시드
    random_seed = int(row["random_seed"])

    # 유고 발생 지점

    incident_location = row["incident_location"]
    locations = [x.strip() for x in incident_location.split(",")] # [본선1, 본선2]
    location = locations[0]

    shutil.copy(network_full_path, copy_network_path)

    # XML 로드
    tree = ET.parse(copy_network_path)
    root = tree.getroot()


    # 본선, 유입램프 교통량
    for vehicle_input in root.findall(".//vehicleInput"):
        input_name = vehicle_input.get("name")
        vehicleInput = vehicle_input.find("timeIntVehVols")

        if vehicleInput is None:
            continue
        for vol in vehicleInput.findall("timeIntervalVehVolume"):
            if input_name == "main":
                vol.set("volume", str(Q_main_in_vehph))
            elif input_name in ["ramp1-1", "ramp1-2"]:
                vol.set("volume", str(Q_ramp_in_per_ramp))

    # 본선/ 램프 유입 중차량 비율
    for vehicle_comp in root.findall(".//vehicleComposition"):
        if vehicle_comp.get("name") == "main":
            relflow_100 = V1_main_in_ratio
            relflow_300 = V2_main_in_ratio
            relflow_630 = V3_main_in_ratio
            relflow_640 = V4_main_in_ratio
            relflow_650 = V5_main_in_ratio

        elif vehicle_comp.get("name") == "ramp":
            relflow_100 = V1_ramp_in_ratio
            relflow_300 = V2_ramp_in_ratio
            relflow_630 = V3_ramp_in_ratio
            relflow_640 = V4_ramp_in_ratio
            relflow_650 = V5_ramp_in_ratio
        else:
            continue
        # <vehCompRelFlows> 태그 찾기
        vehCompRelFlows = vehicle_comp.find("vehCompRelFlows")
        if vehCompRelFlows is not None:
            # <vehicleCompositionRelativeFlow> 태그 찾기
            for relflow in vehCompRelFlows.findall("vehicleCompositionRelativeFlow"):
                vehType = relflow.get("vehType")
                if vehType == "100": # 승용차
                    relflow.set("relFlow", str(relflow_100))
                elif vehType == "300": # 버스
                    relflow.set("relFlow", str(relflow_300))
                elif vehType == "630": # 소형화물
                    relflow.set("relFlow", str(relflow_630))
                elif vehType == "640": # 중형화물
                    relflow.set("relFlow", str(relflow_640))
                elif vehType == "650": # 대형화물
                    relflow.set("relFlow", str(relflow_650))
    # 램프 유출 비율
    for routing_decision  in root.findall(".//vehicleRoutingDecisionStatic"):
        decision_no = routing_decision.get("no")

        # decision별 Q값 매핑
        if decision_no == "1":
            Q = Q1_ramp_out_vehph
        elif decision_no == "2":
            Q = Q2_ramp_out_vehph
        elif decision_no == "3":
            Q = Q3_ramp_out_vehph
        else:
            continue

        veh_routes = routing_decision.find("vehRoutSta")
        if veh_routes is None:
            continue

        for route in veh_routes.findall("vehicleRouteStatic"):
            route_no = route.get("no")

            if route_no == "1":  # 직진
                route.set("relFlow", f"2 0:{round(1 - Q, 3)}")

            elif route_no == "2":  # 진출
                route.set("relFlow", f"2 0:{round(Q, 3)}")

    # 유고 발생 위치 조정
    for rsa in root.findall(".//reducedSpeedArea"):
        name = rsa.get("name")
        if name in locations:
            # 유고 지속시간 설정
            rsa.set("timeFrom", str(acc_start_time))
            rsa.set("timeTo", str(int(acc_start_time + incident_duration_min * 60)))

    # 랜덤시드 설정
    for rds in root.findall(".//simulation"):
        rds.set("randSeed", str(random_seed))


    print(
        f"입력값 | main={Q_main_in_vehph}, ramp={Q_ramp_in_per_ramp}, "
        f"duration={incident_duration_min}, lane={lane_closure_count}, seed={random_seed}, locations={locations}"
    )


    tree.write(copy_network_path, encoding="utf-8", xml_declaration=True)

    # VISSIM 네트워크 로드
    Vissim.LoadNet(copy_network_path)
    Vissim.Graphics.CurrentNetworkWindow.SetAttValue('QuickMode', 1) # 퀵 모드 활성화

    print("VISSIM 시뮬레이션 실행 중...")
    Vissim.Simulation.RunContinuous()
    print("VISSIM 시뮬레이션 실행 완료")

Vissim.Simulation.Stop()
Vissim.Exit()
print("VISSIM 시뮬레이션 종료")
print("==============================================================================================================")



입력값 | main=5490, ramp=1232, duration=60, lane=2, seed=6204553, locations=['본선부128-1', '본선부128-2']
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
입력값 | main=3629, ramp=951, duration=35, lane=2, seed=5649948, locations=['본선부77-1', '본선부77-2']
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
입력값 | main=3464, ramp=1445, duration=60, lane=1, seed=2779261, locations=['진출부6-2']
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
입력값 | main=1511, ramp=1230, duration=10, lane=2, seed=8159263, locations=['본선부29-1', '본선부29-2']
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
입력값 | main=7679, ramp=1125, duration=15, lane=2, seed=9863959, locations=['진출부12-1', '진출부12-2']
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
입력값 | main=3502, ramp=369, duration=45, lane=1, seed=9910251, locations=['진입부2-2']
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
입력값 | main=5762, ramp=1361, duration=75, lane=2, seed=6957800, locations=['본선부17-1', '본선부17-2']
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
입력값 | main=4562, ramp=588, duration=10, lane=1, seed=5145899, locations=

com_error: (-2147352567, '예외가 발생했습니다.', (0, None, None, None, 0, -2147467259), None)